In [60]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Literal
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
from typing import List,Annotated
import operator

load_dotenv()
import os


In [61]:
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview",api_key=os.getenv('GEMINI_API_KEY'))


In [62]:
class SentimentSchema(BaseModel):
    sentimate : Literal['Positive','Negative'] = Field(description='Sentiment of review')


class DiagnosisSchema(BaseModel):
    issue_type : Literal['UX','Performance','Bug','Support','Other'] = Field(description='The Category of issue mentioned in the review')
    tone : Literal['angry','frustrated','disappointed','calm'] = Field(description='The emotional tone expressed by user')
    urgency : Literal['low','medium','high'] = Field(description='How urgent or critical the issue appears to be')




In [63]:
structure_llm = llm.with_structured_output(SentimentSchema)

diagnosis_llm = llm.with_structured_output(DiagnosisSchema)

In [64]:
prompt = 'what is sentimate of review - The software is very bad'
# res = structure_llm.invoke(prompt)
# res.sentimate

In [65]:
class ReviewState(BaseModel):
    review : str | None = None
    sentimate : Literal['Positive','Negative'] | None = Field(description='Sentiment of review',default=None)
    diagnosis : dict | None = None
    res : str | None = None

In [66]:
def fs(state:ReviewState):
    res = structure_llm.invoke(state.review)
    return {'sentimate':res.sentimate}


def check_sentiment(state:ReviewState):
    if state.sentimate == 'Positive':
        return 'positive_res'
    else:
        return 'run_diagnosis'
    

def pr(state:ReviewState):
    prompt = f"""
    Write ONLY ONE short warm thank you message for this review:

    {state.review}

    Also ask user to leave feedback on website.
    Do NOT give multiple options.
    """    
    res = llm.invoke(prompt)

    return {'res':res.content[0]['text']}

def rd(state:ReviewState):
    prompt = f"Diagnose this negative revivew\n\n{state.review}\nand return issue_type, tone and urgency"

    res = diagnosis_llm.invoke(prompt)

    return {'diagnosis':res.model_dump()}

def nr(state:ReviewState):
    d= state.diagnosis
    prompt = f"""
    You are a customer support assistant.

    User issue type: {d['issue_type']}
    Tone: {d['tone']}
    Urgency: {d['urgency']}

    Write ONLY ONE short empathetic resolution message.

    Rules:
    - Do NOT give multiple options
    - Do NOT explain anything
    - Do NOT add headings
    - Keep it professional and helpful
    """

    res = llm.invoke(prompt)

    return {'res':res.content[0]['text']}


    



In [67]:
graph = StateGraph(ReviewState)

graph.add_node('fs',fs)
graph.add_node('pr',pr)
graph.add_node('rd',rd)
graph.add_node('nr',nr)

graph.add_edge(START,'fs')
graph.add_conditional_edges('fs',check_sentiment,{'positive_res':'pr','run_diagnosis':'rd'})
graph.add_edge('rd','nr')
graph.add_edge('nr',END)
graph.add_edge('pr',END)


workflow = graph.compile()

In [68]:
re = input('Enter Review: ')
workflow.invoke({'review':re})

{'review': 'This is extremely frustrating! The app crashes constantly and I lost my data. Fix this immediately or I’m switching to another tool.',
 'sentimate': 'Negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'res': 'I am very sorry for the frustration this has caused, and I have prioritized your case for an immediate fix to resolve this as quickly as possible.'}